<a href="https://colab.research.google.com/github/wuhao007/haowu999/blob/main/metals_haowu999.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 贵金属 AHR999 策略拟合 (黄金 & 白银)
本 Notebook 将 AHR999 指数逻辑应用于黄金和白银，通过对数回归和 MA200 评估价格偏离度。

In [ ]:
!pip install yfinance --quiet
import datetime
import numpy as np
import pandas as pd
import math
from sklearn.linear_model import LinearRegression
import yfinance as yf
import matplotlib.pyplot as plt

In [ ]:
class Metal999:
    def __init__(self, ticker, start_date='2000-01-01'):
        self.ticker = ticker
        self.start_date = pd.to_datetime(start_date)
        self.prices = None
        self.w = None
        self.b = None
        
    def load_data(self):
        print(f"Fetching data for {self.ticker}...")
        # Gold: GC=F (Futures) or GLD (ETF)
        # Silver: SI=F (Futures) or SLV (ETF)
        df = yf.download(self.ticker, start='2000-01-01', progress=False)
        df = df.reset_index()
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df = df[['Date', 'Close']].copy()
        df.columns = ['Date', 'Close']
        self.prices = df.dropna()
        self._fit_model()
        return self.prices

    def _fit_model(self):
        df = self.prices.copy()
        # 计算距离 1970 或自定义起点的天数（贵金属拟合建议用较长跨度）
        df['Days'] = (df['Date'] - self.start_date).dt.days
        df = df[df['Days'] > 0].copy()
        
        x = np.log10(df['Days'].values).reshape(-1, 1)
        y = np.log10(df['Close'].values)
        
        model = LinearRegression().fit(x, y)
        self.w = model.coef_[0]
        self.b = model.intercept_
        self.score = model.score(x, y)
        print(f"{self.ticker} 拟合完成: w={self.w:.4f}, R²={self.score:.4f}")

    def calculate_ahr999(self):
        df = self.prices.copy()
        df['MA200'] = df['Close'].rolling(200).mean()
        df['Days'] = (df['Date'] - self.start_date).dt.days
        df['FitPrice'] = 10 ** (self.w * np.log10(df['Days']) + self.b)
        # AHR999 = (价格/MA200) * (价格/拟合价)
        df['AHR999'] = (df['Close'] / df['MA200']) * (df['Close'] / df['FitPrice'])
        return df.dropna()

In [ ]:
def analyze_metal(ticker):
    m = Metal999(ticker)
    df = m.load_data()
    res = m.calculate_ahr999()
    
    current = res.iloc[-1]
    
    # 重点：计算贵金属的历史百分位阈值
    # 因为贵金属波动小，0.45 可能几十年才一次，我们需要动态计算
    bottom_threshold = res['AHR999'].quantile(0.10) # 历史最便宜的 10%
    invest_threshold = res['AHR999'].quantile(0.50) # 历史平均线
    
    print(f"\n--- {ticker} 实时分析 ---")
    print(f"当前价格: {current['Close']:.2f}")
    print(f"AHR999 值: {current['AHR999']:.4f}")
    print(f"建议抄底线 (10%分位): {bottom_threshold:.4f}")
    print(f"建议定投线 (50%分位): {invest_threshold:.4f}")
    
    # 绘图
    plt.figure(figsize=(12, 5))
    plt.plot(res['Date'], res['AHR999'], label='AHR999 Index')
    plt.axhline(y=bottom_threshold, color='r', linestyle='--', label='Bottom (10%)')
    plt.axhline(y=invest_threshold, color='g', linestyle='--', label='Invest (50%)')
    plt.title(f"{ticker} AHR999 Index Trend")
    plt.yscale('log')
    plt.legend()
    plt.show()

analyze_metal('GC=F') # 黄金期货
analyze_metal('SI=F') # 白银期货